# NLP Masterclass 02: Word Embeddings & Word2Vec

**Track:** Natural Language Processing (0 to Masterclass)  
**Prerequisites:** NLP 01 (Tokenization), ML Fundamentals  
**Difficulty:** ⭐⭐ Intermediate  
**Estimated Time:** 90–120 minutes

---

## 🎓 Socratic Deep Check

> *"Word2Vec showed that king - man + woman ≈ queen. These embeddings capture semantic meaning as geometry. But Word2Vec gives 'bank' the SAME vector whether it means 'river bank' or 'financial bank'. How do contextual embeddings (BERT, GPT) solve this fundamental limitation?"*

---

## Why Embeddings Are the Foundation of Modern AI

Computers understand numbers, not words. **Embeddings convert words into dense vectors** where similar words live close together. This single idea powers:
- **RAG** (NB28): Embed documents, find similar ones
- **Vector databases** (NB27): Store and search embeddings
- **LLM internals**: Every token in GPT starts as an embedding

## Table of Contents
1. From One-Hot to Dense Vectors
2. Word2Vec: Skip-Gram from Scratch
3. Vector Arithmetic & Analogies
4. Pre-trained Embeddings (GloVe, FastText)
5. From Static to Contextual Embeddings

---

# NLP Masterclass 02: Word Embeddings & Word2Vec

**Track:** Natural Language Processing (0 to Masterclass)  
**Prerequisites:** NLP 01 (Tokenization), ML Fundamentals  
**Difficulty:** ⭐⭐ Intermediate  
**Estimated Time:** 120–150 minutes

---

## 🎓 Socratic Deep Check

> *"J.R. Firth famously said, 'You shall know a word by the company it keeps.' If every word's meaning is defined by its neighbors, then synonymity becomes a geometric property. But Word2Vec gives 'bank' the same vector regardless of context. How does this 'static' limitation motivate the jump to Transformers, and why do we still use Word2Vec for specialized tasks today?"*

---

## The Core Problem: How Machines "Read"

Computers are calculation engines for linear algebra, not linguists. For a model to process language, we must map discrete tokens into a continuous vector space $\mathbb{R}^d$. This notebook explores the **Representation Learning** revolution that changed NLP forever: the shift from sparse, high-dimensional counts to dense, low-dimensional embeddings.

### Table of Contents
1. **The Distributional Hypothesis**: Theory and Foundations.
2. **The Word2Vec Revolution**: CBOW vs. Skip-Gram architectures.
3. **From Scratch in PyTorch**: Implementing Skip-Gram with Negative Sampling (SGNS).
4. **GloVe (Global Vectors)**: Bridging count-based and predictive models.
5. **FastText**: Solving the Out-of-Vocabulary (OOV) problem with subwords.
6. **Evaluation**: How to measure the quality of a vector space.

---

In [ ]:
!pip install -q torch numpy matplotlib scikit-learn gensim tqdm

## 1 · The Distributional Hypothesis

### Localist vs. Distributed Representations

In a **Localist** representation (like One-Hot Encoding), each word is an independent dimension. 
- **Curse of Dimensionality**: A 100k-word vocabulary requires 100k-dimensional vectors.
- **Orthogonality**: All words are equidistant. `cosine(cat, dog) = 0`, the same as `cosine(cat, refrigerator)`.

A **Distributed** representation (Embedding) represents words as dense vectors where meaning is "distributed" across multiple dimensions.

### The Distributional Hypothesis
The bedrock of modern NLP is the **Distributional Hypothesis**: Words that occur in similar contexts tend to have similar meanings. 

> "A word is characterized by the company it keeps." — J.R. Firth (1957)

If we can predict a word's context, we can "learn" its meaning implicitly. This is the foundation of **Self-Supervised Learning**.

--- 
## 2 · The Word2Vec Revolution

In 2013, Mikolov et al. introduced two architectures that efficiently learned word vectors from massive datasets.

### A. Continuous Bag-of-Words (CBOW)
**Task**: Predict the *center* word given the surrounding *context* words.
- Example: `"The [?] sat on the mat"` $\rightarrow$ predict `"cat"`.
- Good for frequent words; faster to train.

### B. Skip-Gram
**Task**: Predict the *context* words given the *center* word.
- Example: `"cat"` $\rightarrow$ predict `["The", "sat", "on"]`.
- Better at representing rare words and capturing complex relationships.

### The Mathematical Objective
The standard objective function for Skip-Gram is to maximize the log-likelihood of the context words $w_{t+j}$ given the center word $w_t$:

$$J(\theta) = \frac{1}{T} \sum_{t=1}^T \sum_{-m \le j \le m, j \neq 0} \log P(w_{t+j} | w_t)$$

The probability is typically modeled via **Softmax**:

$$P(o | c) = \frac{\exp(u_o^T v_c)}{\sum_{w=1}^V \exp(u_w^T v_c)}$$

**The Problem:** The denominator requires summing over the entire vocabulary $V$ (e.g., 500,000 words), making it computationally impossible for large datasets.

### The Solution: Negative Sampling (SGNS)
Instead of predicting the exact word (Multi-class Classification), we turn it into a **Binary Classification** task:
1. **Positive Pair**: (center_word, actual_context_word) $\rightarrow$ Predict 1
2. **Negative Pairs**: (center_word, random_noise_word) $\rightarrow$ Predict 0

The new objective function becomes:

$$J_{SGNS} = \log \sigma(u_o^T v_c) + \sum_{i=1}^k \mathbb{E}_{w_i \sim P_n(w)} [\log \sigma(-u_i^T v_c)]$$

where $\sigma$ is the sigmoid function and $k$ is the number of negative samples.

--- 
## 3 · Skip-Gram with Negative Sampling (PyTorch from Scratch)

We will implement a minimal version of SGNS to understand the mechanics of embedding updates.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import random
from collections import Counter

# 1. Simple Data Preparation
text = """The quick brown fox jumps over the lazy dog. 
          Data science is an interdisciplinary field that uses scientific methods, 
          processes, algorithms and systems to extract knowledge and insights 
          from noisy, structured and unstructured data."""

tokens = text.lower().replace('.', '').split()
vocab = {w: i for i, w in enumerate(sorted(set(tokens)))}
idx_to_word = {i: w for w, i in vocab.items()}
vocab_size = len(vocab)
word_counts = Counter(tokens)

# Negative Sampling Distribution (P(w)^3/4 as per the paper)
freqs = np.array([word_counts[idx_to_word[i]] for i in range(vocab_size)])
unigram_dist = freqs**0.75 / np.sum(freqs**0.75)

class Word2VecDataset(Dataset):
    def __init__(self, tokens, vocab, window_size=2, n_negs=5):
        self.pairs = []
        for i, token in enumerate(tokens):
            center = vocab[token]
            for j in range(max(0, i-window_size), min(len(tokens), i+window_size+1)):
                if i != j:
                    self.pairs.append((center, vocab[tokens[j]]))
        self.n_negs = n_negs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        center, context = self.pairs[idx]
        # Sample negative tokens
        negs = torch.multinomial(torch.from_numpy(unigram_dist), self.n_negs, replacement=True)
        return torch.tensor(center), torch.tensor(context), negs

# 2. The Skip-Gram Model
class SGNSModel(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        # Each word has an 'input' vector and an 'output' vector
        self.in_embed = nn.Embedding(vocab_size, embed_dim)
        self.out_embed = nn.Embedding(vocab_size, embed_dim)
        
        # Initialize with small random values
        self.in_embed.weight.data.uniform_(-0.5/embed_dim, 0.5/embed_dim)
        self.out_embed.weight.data.uniform_(0, 0)

    def forward(self, center, context, negative):
        # center: (batch)
        # context: (batch)
        # negative: (batch, n_negs)
        
        v_c = self.in_embed(center)   # (batch, embed_dim)
        u_o = self.out_embed(context) # (batch, embed_dim)
        u_n = self.out_embed(negative) # (batch, n_negs, embed_dim)

        # Positive pair dot product: want to maximize
        pos_score = torch.sum(v_c * u_o, dim=1) # (batch)
        pos_loss = -torch.log(torch.sigmoid(pos_score) + 1e-10).mean()

        # Negative pairs dot product: want to minimize
        # (batch, 1, dim) @ (batch, dim, n_negs) -> (batch, 1, n_negs)
        neg_score = torch.bmm(u_n, v_c.unsqueeze(2)).squeeze(2) # (batch, n_negs)
        neg_loss = -torch.log(torch.sigmoid(-neg_score) + 1e-10).sum(1).mean()

        return pos_loss + neg_loss

# 3. Training Loop
embed_dim = 16
model = SGNSModel(vocab_size, embed_dim)
optimizer = optim.Adam(model.parameters(), lr=0.01)
dataset = Word2VecDataset(tokens, vocab)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

print("Training SGNS...")
for epoch in range(100):
    total_loss = 0
    for center, context, negs in dataloader:
        optimizer.zero_grad()
        loss = model(center, context, negs)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {total_loss/len(dataloader):.4f}")

### # TEST IT: Verification
Let's see if semantically related words are close in our tiny vector space.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def get_similarity(w1, w2):
    v1 = model.in_embed.weight[vocab[w1]].detach().numpy()
    v2 = model.in_embed.weight[vocab[w2]].detach().numpy()
    return cosine_similarity([v1], [v2])[0][0]

print(f"Similarity (data, knowledge): {get_similarity('data', 'knowledge'):.3f}")
print(f"Similarity (data, fox):       {get_similarity('data', 'fox'):.3f}")
print(f"Similarity (quick, brown):    {get_similarity('quick', 'brown'):.3f}")

--- 
## 4 · GloVe: Global Vectors

While Word2Vec is a **predictive** model (iterates over local windows), **GloVe** (Pennington et al., 2014) is a **count-based matrix factorization** model.

### The Key Difference
- **Word2Vec Strategy**: Scan text window by window. Fails to leverage global statistics directly.
- **GloVe Strategy**: Build a global **Co-occurrence Matrix** $X$ first. Then factorize it to get embeddings.

### The GloVe Objective
GloVe minimizes the difference between the dot product of two vectors and the log of their co-occurrence frequency, weighted by a function $f(X_{ij})$ to prevent common words from dominating:

$$J = \sum_{i,j=1}^V f(X_{ij}) (w_i^T \tilde{w}_j + b_i + \tilde{b}_j - \log X_{ij})^2$$

**Why use GloVe?**
- Better at capturing global semantic relationships.
- Often more stable for smaller datasets than Word2Vec.

--- 
## 5 · FastText: Handling Subwords & OOV

Word2Vec and GloVe treat words as atomic units. If a word wasn't in the training set (e.g., "antigravity"), they have no vector for it. This is the **Out-of-Vocabulary (OOV)** problem.

**FastText** (Bojanowski et al., 2016) solves this by treating each word as a bag of **character n-grams**.

### Example: "where" with n=3
- Subwords: `<wh`, `whe`, `her`, `ere`, `re>` (plus the word itself `<where>`)
- The vector for "where" is the **sum** of these subword vectors.

**Advantages:**
1. **OOV Support**: Even if the model hasn't seen "antigravity", it has seen "anti-" and "-gravity". It can construct a reasonable vector by summing subwords.
2. **Morphology**: It naturally captures relationships between "eat", "eats", and "eating" because they share character n-grams.

In [ ]:
from gensim.models import FastText

# Create a small toy dataset
sentences = [["the", "speed", "of", "light", "is", "constant"], 
             ["gravity", "bends", "the", "fabric", "of", "space"]]

# Train FastText
ft_model = FastText(sentences, vector_size=10, window=3, min_count=1, epochs=10)

# TEST OOV: "spacetime" was NOT in the training data
try:
    vector = ft_model.wv['spacetime']
    print("Successfully generated vector for OOV word 'spacetime'!")
    print(f"Vector (first 5 dims): {vector[:5]}")
except KeyError:
    print("FastText failed (this shouldn't happen!)")

--- 
## 6 · Intrinsic Evaluation

How do we know if our embeddings are "good"? We use two main approaches:

### 1. Word Similarity
Compare the model's similarity score with human-labeled datasets like **Wordsim-353**. If humans rate (cat, dog) at 9/10 and our cosine similarity is high, the model is learning well.

### 2. Analogies (Vector Arithmetic)
The most famous test of vector spaces is solving analogies: $A \text{ is to } B \text{ as } C \text{ is to } X$.

$$\text{vec(king)} - \text{vec(man)} + \text{vec(woman)} \approx \text{vec(queen)}$$

This works because the vector space encodes semantic **relations** as geometric **offsets**.

--- 
## ✅ Knowledge Check

### Q1: The Softmax Problem
Why does the standard Word2Vec objective function require Negative Sampling or Hierarchical Softmax for production use?

<details><summary>👉 Answer</summary>
Standard Softmax requires calculating the dot product for EVERY word in the vocabulary for every training step. With a vocab of 500k+, this is computationally prohibitive. Negative Sampling simplifies this into a binary task against a handful of noise words, reducing complexity from O(V) to O(k), where k is the number of negative samples (usually 5-20).
</details>

### Q2: FastText vs. Word2Vec
In a production environment for a highly technical domain (e.g., Medicine or Chemistry) where many rare/unseen words appear, which embedding model should you choose?

<details><summary>👉 Answer</summary>
FastText. Technical domains often have complex morphology (e.g., 'dihydrogen monoxide'). FastText's subword information allows it to generalize to unseen chemical or medical terms by leveraging prefixes and suffixes used in known words.
</details>

### Q3: When to use Static Embeddings?
If Transformers (BERT/GPT) provide superior contextual embeddings, why would an engineer ever use static embeddings like Word2Vec today?

<details><summary>👉 Answer</summary>
1. **Inference Speed**: Table lookups for static embeddings are nearly instantaneous. Transformers require massive GPU compute.
2. **Dimensionality**: Word2Vec (100-300D) is much smaller than modern embeddings (768-4096D), making it cheaper for large-scale storage/similarity search.
3. **Baselines**: For simple classification tasks, Word2Vec + a Linear layer is a powerful baseline that avoids the overhead of LLMs.
</details>

--- 
## 🔨 Practical Practice

1. **Modify the SGNS Model**: Add a learning rate scheduler and observe the effect on loss convergence.
2. **The Analogy Explorer**: Find a word pair in the `gensim` pre-trained models (e.g., `glove-wiki-gigaword-100`) that demonstrates a geographical analogy (e.g., `Paris - France + Germany = Berlin`).
3. **Bias Analysis**: Calculate the similarity between `doctor` and `man` vs `doctor` and `woman` in a pre-trained Word2Vec model. Reflect on how historical data bias manifests in the representation space.

**Next →** NLP 03: Recurrent Networks & LSTMs